# SABR and Bartlett Deltas

Philipp Kreiter, IJsbrand Meeter

### 0 - Imports and Constants

In [25]:
import numpy as np
import pandas as pd
from scipy import stats, optimize
import matplotlib.pyplot as plt

BETA1 = 0.5826 # barrrier adjustment
MONEYNESS_BIN_EDGES = np.array([0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.95])


### 1 - Data loading and cleaning

In [26]:
def load_and_preprocess_data(filename: str = ""):
    assert filename, "Please provide a filename to load the data from."

    # load csv
    df = pd.read_csv(filename)
    
    print(f"Row count before filtering: {len(df)}")

    # keep call options only
    df = df[df['cp_flag'] == 'C']

    print(f"Row count before filtering: {len(df)}")

    # convert date and exdate to datetime
    df['date'] = pd.to_datetime(df['date'], format='ISO8601')
    df['exdate'] = pd.to_datetime(df['exdate'], format='ISO8601')

    # rescale strike price in index points
    df['strike_price'] = df['strike_price'] / 1000
    # construct midquote V_t
    df['midquote'] = (df['best_bid'] + df['best_offer']) / 2

    # drop rows with missing implied volatility
    missing_iv_before = df['impl_volatility'].isna().sum()
    df = df[df['impl_volatility'] > 0.0]
    print(f"Missing implied volatility rows: {missing_iv_before}")
    print(f"Missing V_t rows: {df['midquote'].isna().sum()}")
    print(f"missing delta rows: {df['delta'].isna().sum()}")
    
    # drop not needed columns
    keep_cols = [
    'date', 'exdate', 'symbol', 'strike_price', 'midquote', 'impl_volatility', 'delta'
    ]
    df = df[keep_cols]



    # unique days
    print(f"Unique trading days: {df['date'].nunique()}")
    print(f"Unique expiration dates: {df['exdate'].nunique()}")

    # final column list 
    print("Columns in the dataframe:", df.columns.tolist())

    return df

print("---------------------------------")
print("28-02-2023 DATA")
df_day = load_and_preprocess_data("option20230228.csv")
print("---------------------------------")
print("01-02-2023 to 28-02-2023 DATA")
df_period = load_and_preprocess_data("option20230201_20230228.csv")




---------------------------------
28-02-2023 DATA
Row count before filtering: 18382
Row count before filtering: 9191
Missing implied volatility rows: 373
Missing V_t rows: 0
missing delta rows: 0
Unique trading days: 1
Unique expiration dates: 49
Columns in the dataframe: ['date', 'exdate', 'symbol', 'strike_price', 'midquote', 'impl_volatility', 'delta']
---------------------------------
01-02-2023 to 28-02-2023 DATA
Row count before filtering: 175301
Row count before filtering: 175301
Missing implied volatility rows: 21573
Missing V_t rows: 0
missing delta rows: 0
Unique trading days: 19
Unique expiration dates: 67
Columns in the dataframe: ['date', 'exdate', 'symbol', 'strike_price', 'midquote', 'impl_volatility', 'delta']


### 2 - Contract Time Series and Delta V